# SmartRetail – AI Demand Forecasting & Inventory Analytics
### Data Science Module | Python | Pandas | scikit-learn
> Analyzing retail sales data to predict demand, detect slow-moving products, and surface inventory insights.

## Section 1 – Setup & Generate Retail Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('outputs/charts', exist_ok=True)

# Generate realistic retail sales data
np.random.seed(42)
n = 5000
categories = ['Groceries', 'Beverages', 'Snacks', 'Dairy', 'Frozen', 'Personal Care', 'Household']
outlets = ['Small', 'Medium', 'Large', 'Supermarket']
locations = ['Urban', 'Suburban', 'Rural']

df = pd.DataFrame({
    'Date': pd.date_range('2022-01-01', periods=n, freq='H'),
    'Product_Category': np.random.choice(categories, n),
    'Outlet_Size': np.random.choice(outlets, n),
    'Location_Type': np.random.choice(locations, n),
    'Item_Visibility': np.random.uniform(0.01, 0.35, n),
    'Item_MRP': np.random.uniform(30, 500, n),
    'Item_Sales': np.random.normal(2200, 800, n).clip(100)
})

# Add time features
df['month'] = df['Date'].dt.month
df['day_of_week'] = df['Date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)
df['quarter'] = df['Date'].dt.quarter

# Seasonal boost (festival months + weekends)
df.loc[df['month'].isin([10,11,12]), 'Item_Sales'] *= 1.4
df.loc[df['is_weekend'] == 1, 'Item_Sales'] *= 1.15

sales_col = 'Item_Sales'
print('Dataset created:', df.shape)
df.head()

## Section 2 – Data Cleaning

In [ ]:
print('Missing values:\n', df.isnull().sum())

for col in df.select_dtypes(include=np.number).columns:
    df[col].fillna(df[col].median(), inplace=True)

for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

print('\nAfter cleaning - Shape:', df.shape)
print(df.dtypes)

## Section 3 – Feature Engineering

In [ ]:
df['rolling_7day_avg'] = df[sales_col].rolling(7, min_periods=1).mean()
df['rolling_30day_avg'] = df[sales_col].rolling(30, min_periods=1).mean()
df['sales_lag_1'] = df[sales_col].shift(1).fillna(df[sales_col].mean())
df['sales_lag_7'] = df[sales_col].shift(7).fillna(df[sales_col].mean())

print('New features added: rolling_7day_avg, rolling_30day_avg, sales_lag_1, sales_lag_7')
print('Shape:', df.shape)
df.head()

## Section 4 – EDA & Visualizations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('SmartRetail – Sales Analytics Dashboard', fontsize=16, fontweight='bold')

# Chart 1: Sales Distribution
axes[0,0].hist(df[sales_col], bins=40, color='steelblue', edgecolor='white')
axes[0,0].set_title('Sales Distribution')
axes[0,0].set_xlabel('Sales')

# Chart 2: Monthly Sales Trend
monthly = df.groupby('month')[sales_col].mean()
axes[0,1].bar(monthly.index, monthly.values, color='teal')
axes[0,1].set_title('Average Sales by Month (Seasonality)')
axes[0,1].set_xlabel('Month')

# Chart 3: Rolling Average Trend
axes[0,2].plot(df[sales_col].values[:300], label='Actual', alpha=0.5)
axes[0,2].plot(df['rolling_7day_avg'].values[:300], label='7-day avg', color='red', linewidth=2)
axes[0,2].set_title('Sales Trend with Rolling Average')
axes[0,2].legend()

# Chart 4: Sales by Category
cat_sales = df.groupby('Product_Category')[sales_col].mean().sort_values(ascending=False)
cat_sales.plot(kind='bar', ax=axes[1,0], color='coral', edgecolor='white')
axes[1,0].set_title('Avg Sales by Product Category')
axes[1,0].tick_params(axis='x', rotation=45)

# Chart 5: Weekday vs Weekend
df.groupby('is_weekend')[sales_col].mean().plot(kind='bar', ax=axes[1,1], color=['steelblue','orange'])
axes[1,1].set_title('Weekday vs Weekend Sales')
axes[1,1].set_xticklabels(['Weekday', 'Weekend'], rotation=0)

# Chart 6: Slow-Moving Products
df['is_slow_moving'] = (df[sales_col] < df[sales_col].quantile(0.25)).astype(int)
df['is_slow_moving'].value_counts().plot(kind='pie', ax=axes[1,2],
    labels=['Normal', 'Slow-Moving'], colors=['#2ecc71','#e74c3c'], autopct='%1.1f%%')
axes[1,2].set_title('Slow-Moving Product Share')
axes[1,2].set_ylabel('')

plt.tight_layout()
plt.savefig('outputs/charts/sales_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Charts saved!')

## Section 5 – ML Model (Random Forest Demand Forecasting)

In [ ]:
df_model = df.copy()
le = LabelEncoder()

for col in df_model.select_dtypes(include='object').columns:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

df_model.drop('Date', axis=1, inplace=True)

X = df_model.drop(sales_col, axis=1)
y = df_model[sales_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f'RMSE : {rmse:.2f}')
print(f'MAE  : {mae:.2f}')
print(f'R²   : {r2:.4f}')

## Section 6 – Feature Importance & Business Insights

In [ ]:
feat_imp = pd.Series(model.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
feat_imp.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('SmartRetail – Top Demand Drivers (Feature Importance)')
plt.ylabel('Importance Score')
plt.tight_layout()
plt.savefig('outputs/charts/feature_importance.png', dpi=150)
plt.show()

slow = df[df['is_slow_moving'] == 1]
print(f'\n📊 Business Insights:')
print(f'1. {len(slow):,} out of {len(df):,} records are slow-moving — candidates for discount or combo offers.')
print(f'2. Top demand driver: {feat_imp.index[0]} — prioritize stocking decisions around this feature.')
print(f'3. Model R² = {r2:.2f} — the forecasting model explains {r2*100:.0f}% of sales variance.')
print(f'4. Festival months (Oct-Dec) drive ~40% higher sales — increase stock buffer by Q3.')
print(f'5. Weekend sales are ~15% higher — schedule restocking on Thursdays/Fridays.')

---
## 📝 Resume Description
> Built the data science core of **SmartRetail**, an AI-powered retail inventory system, covering demand forecasting, seasonal trend analysis, slow-moving product detection, and inventory anomaly insights using Python, Pandas, and scikit-learn on 5,000+ synthetic retail transactions.

**Tools:** Python | Pandas | NumPy | Matplotlib | Seaborn | scikit-learn (Random Forest) | Jupyter